# O problema
Imagine agora que você foi contratado(a) como Expert em Data Analytics por um grande hospital para entender como foi o comportamento da população na época da pandemia da COVID-19 e quais indicadores seriam importantes para o planejamento, caso haja um novo surto da doença.

Apesar de ser contratado(a) agora, a sua área observou que a utilização do estudo do PNAD-COVID 19 do IBGE seria uma ótima base para termos boas respostas ao problema proposto, pois são dados confiáveis. Porém, não será necessário utilizar todas as perguntas realizadas na pesquisa para enxergar todas as oportunidades ali postas.

É sempre bom ressaltar que há dados triviais que precisam estar no projeto, pois auxiliam muito na análise dos dados:
*   Características clínicas dos sintomas;
*   Características da população;
*   Características econômicas da sociedade.

O Head de Dados pediu para que você entrasse na base de dados do PNAD-COVID-19 do IBGE (https://covid19.ibge.gov.br/pnad-covid/) e organizasse esta base para análise, utilizando Banco de Dados em Nuvem e trazendo as seguintes características:
1. Utilização de no máximo 20 questionamentos realizados na pesquisa;
2. Utilizar 3 meses para construção da solução;
3. Caracterização dos sintomas clínicos da população;
4. Comportamento da população na época da COVID-19;
5. Características econômicas da Sociedade;

Seu objetivo será trazer uma breve análise dessas informações, como foi a organização do banco, as perguntas selecionadas para a resposta do problema e quais seriam as principais ações que o hospital deverá tomar em caso de um novo surto de COVID-19.

Dica: leiam com atenção a base de dados e toda a documentação que o site o PNAD – Covid19 traz, principalmente os dicionários, que ajudam e muito no entendimento da Base de Dados.

Dica 2: utilizem o que já foi ensinado e consolidado nas outras fases para
apresentar a resolução do projeto.

Lembre-se de que você poderá apresentar o desenvolvimento do seu projeto durante as lives com docentes. Essa é uma boa oportunidade para discutir sobre as dificuldades encontradas e pegar dicas valiosas com especialistas e colegas de turma.

Não se esqueça que isso é um entregável obrigatório! Se atente para o prazo de entrega até o final da fase.
Boa Sorte!

## Importações de libs e início de sessão

In [1]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.functions import countDistinct, col, when, lit
from pyspark.sql.types import *
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: f6900a06-b61c-4327-8323-a1356a2fd55f
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session f6900a06-b61c-4327-8323-a1356a2fd55f to get into ready status...
Session f6900a06-b61c-4327-8323-a1356a2fd55f 

In [2]:
dyf = glueContext.create_dynamic_frame.from_catalog(database='default', table_name='pnad_092020')
dyf.printSchema()

root
|-- Ano: string
|-- UF: string
|-- CAPITAL: string
|-- RM_RIDE: string
|-- V1008: string
|-- V1012: string
|-- V1013: string
|-- V1016: string
|-- Estrato: string
|-- UPA: string
|-- V1022: string
|-- V1023: string
|-- V1030: string
|-- V1031: string
|-- V1032: string
|-- posest: string
|-- A001: string
|-- A001A: string
|-- A001B1: string
|-- A001B2: string
|-- A001B3: string
|-- A002: string
|-- A003: string
|-- A004: string
|-- A005: string
|-- A006: string
|-- A006A: string
|-- A006B: string
|-- A007: string
|-- A007A: string
|-- A008: string
|-- A009: string
|-- B0011: string
|-- B0012: string
|-- B0013: string
|-- B0014: string
|-- B0015: string
|-- B0016: string
|-- B0017: string
|-- B0018: string
|-- B0019: string
|-- B00110: string
|-- B00111: string
|-- B00112: string
|-- B00113: string
|-- B002: string
|-- B0031: string
|-- B0032: string
|-- B0033: string
|-- B0034: string
|-- B0035: string
|-- B0036: string
|-- B0037: string
|-- B0041: string
|-- B0042: string
|-- B004

In [3]:
df_bronze = dyf.toDF()
df_bronze.count()

1149197
/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.


# Limpeza da base

In [4]:
identifying_columns = [
    "Ano", "UF", "CAPITAL", "RM_RIDE", "V1008", "V1012", "V1013", "V1016",
    "Estrato", "UPA", "V1022", "V1023", "V1030", "V1031", "V1032", "posest",
    "A001A", "A001B1", "A001B2", "A001B3", "A002", "A003", "A004", "A005",
    "A006", "A007", "A008", "A009"
]

def count_total_duplicate_records(dataframe, df_name, id_columns):
    # Agrupa o DataFrame pelas colunas de identificação e conta as ocorrências de cada combinação.
    counted_df = dataframe.groupBy(id_columns).count()

    # Filtra por combinações que aparecem mais de uma vez (duplicatas)
    # e soma (contagem - 1) para cada grupo para obter o número total de registros duplicados.
    total_duplicates = counted_df.filter(F.col("count") > 1)\
                                 .withColumn("duplicate_records", F.col("count") - 1)\
                                 .agg(F.sum("duplicate_records").alias("total_duplicate_records"))

    # Pega o resultado do cálculo de duplicatas. Se não houver duplicatas, o resultado é None, então definimos como 0.
    result = total_duplicates.collect()[0]["total_duplicate_records"]
    if result is None:
        result = 0 # Nenhuma duplicata encontrada

    print(f"DataFrame '{df_name}' possui {result} registros duplicados com base nas colunas de identificação.")

print("\n--- Contando registros duplicados para df_bronze ---")
# Chama a função para contar duplicatas no DataFrame df_bronze, usando as colunas identificadoras.
count_total_duplicate_records(df_bronze, "df_bronze", identifying_columns)


--- Contando registros duplicados para df_bronze ---
DataFrame 'df_bronze' possui 2033 registros duplicados com base nas colunas de identificação.


In [5]:
df_silver = df_bronze.dropDuplicates(subset=identifying_columns)

# Verifica as duplicações novamente
count_total_duplicate_records(df_silver, "df_silver", identifying_columns)

DataFrame 'df_silver' possui 0 registros duplicados com base nas colunas de identificação.


## Seleção de colunas inicial

In [6]:
colunas_selecionadas = [
    # Sintomas
    "B0011", "B0012", "B0013", "B0014", "B0015", "B0016",
    "B0017", "B0018", "B0019", "B00110", "B00111",
    "B00112", "B00113",

    # Medidas Adotadas e Histórico Clínico
    "B0031", "B0032", "B0033", "B0035", "B0036", "B0037",          # Providências
    "B0041", "B0042", "B0043", "B0044", "B0045", "B0046",          # Estabelecimentos
    "B005", "B006", "B007",                                        # Internação, Respirador, Plano
    "B0101", "B0102", "B0103", "B0104", "B0105", "B0106",          # Comorbidades (Grupo de risco)
    "B011",                                                        # Isolamento
    "B008", "B009B", "B009D", "B009F",                             # Testagem

    # Perfil Socioeconômico e Demográfico
    "A002", "A003", "A004", "A005",                                # Idade, Sexo, Cor, Escolaridade
    "C001", "C002", "C003", "C013", "C01012",                      # Trabalho, Afastamento, Home office, Renda
    "D0051", "D0061"                                               # Auxílio Covid, Seguro Desemprego
]

df_silver = df_silver.select(colunas_selecionadas)

In [7]:
df_silver.printSchema()

root
 |-- B0011: string (nullable = true)
 |-- B0012: string (nullable = true)
 |-- B0013: string (nullable = true)
 |-- B0014: string (nullable = true)
 |-- B0015: string (nullable = true)
 |-- B0016: string (nullable = true)
 |-- B0017: string (nullable = true)
 |-- B0018: string (nullable = true)
 |-- B0019: string (nullable = true)
 |-- B00110: string (nullable = true)
 |-- B00111: string (nullable = true)
 |-- B00112: string (nullable = true)
 |-- B00113: string (nullable = true)
 |-- B0031: string (nullable = true)
 |-- B0032: string (nullable = true)
 |-- B0033: string (nullable = true)
 |-- B0035: string (nullable = true)
 |-- B0036: string (nullable = true)
 |-- B0037: string (nullable = true)
 |-- B0041: string (nullable = true)
 |-- B0042: string (nullable = true)
 |-- B0043: string (nullable = true)
 |-- B0044: string (nullable = true)
 |-- B0045: string (nullable = true)
 |-- B0046: string (nullable = true)
 |-- B005: string (nullable = true)
 |-- B006: string (nullable = 

In [9]:
df_silver.count()

1147164


# Enriquecimento com o dicionário

In [10]:
# Dicionário de tradução (De Código IBGE para Nome de Negócio Silver)
mapa_renomeacao_silver = {
    # Sintomas
    "B0011": "sintoma_febre",
    "B0012": "sintoma_tosse",
    "B0013": "sintoma_dor_garganta",
    "B0014": "sintoma_dificuldade_respirar",
    "B0015": "sintoma_dor_cabeca",
    "B0016": "sintoma_dor_peito",
    "B0017": "sintoma_nausea",
    "B0018": "sintoma_nariz_entupido",
    "B0019": "sintoma_fadiga",
    "B00110": "sintoma_dor_olhos",
    "B00111": "sintoma_perda_cheiro_sabor",
    "B00112": "sintoma_dor_muscular",
    "B00113": "sintoma_diarreia",

    # Medidas Adotadas e Histórico Clínico
    "B0031": "providencia_ficou_em_casa",
    "B0032": "providencia_ligou_medico",
    "B0033": "providencia_comprou_remedio",
    "B0035": "providencia_visita_sus",
    "B0036": "providencia_visita_privado",
    "B0037": "providencia_outra",
    "B0041": "buscou_posto_sus",
    "B0042": "buscou_pronto_socorro_sus",
    "B0043": "buscou_hospital_sus",
    "B0044": "buscou_ambulatorio_privado",
    "B0045": "buscou_pronto_socorro_privado",
    "B0046": "buscou_hospital_privado",
    "B005": "internacao",
    "B006": "suporte_ventilatorio",
    "B007": "plano_saude",
    "B0101": "comorbidade_diabetes",
    "B0102": "comorbidade_hipertensao",
    "B0103": "comorbidade_asma",
    "B0104": "comorbidade_doenca_cardiaca",
    "B0105": "comorbidade_depressao",
    "B0106": "comorbidade_cancer",
    "B011": "nivel_isolamento",
    "B008": "realizou_teste",
    "B009B": "resultado_swab",
    "B009D": "resultado_teste_rapido",
    "B009F": "resultado_exame_sangue",

    # Perfil Socioeconômico e Demográfico
    "A002": "idade",
    "A003": "sexo",
    "A004": "cor_raca",
    "A005": "escolaridade",
    "C001": "status_trabalho",
    "C002": "motivo_afastamento_temporario",
    "C003": "continuou_remunerado",
    "C013": "trabalho_remoto",
    "C01012": "faixa_renda",
    "D0051": "auxilio_emergencial",
    "D0061": "seguro_desemprego"
}

# Função para renomear colunas de um DataFrame
def rename_columns_for_df(df, mapping):
    for old_col, new_col in mapping.items():
        if old_col in df.columns:  # Garante que a coluna existe antes de tentar renomear
            df = df.withColumnRenamed(old_col, new_col)
    return df

# Renomeia as colunas para o DataFrame 'df_silver'
df_silver = rename_columns_for_df(df_silver, mapa_renomeacao_silver)

# Exibe o schema final para validação
print("Schema de df_silver após renomeação (ou confirmação dos nomes existentes):")
df_silver.printSchema()

Schema de df_silver após renomeação (ou confirmação dos nomes existentes):
root
 |-- sintoma_febre: string (nullable = true)
 |-- sintoma_tosse: string (nullable = true)
 |-- sintoma_dor_garganta: string (nullable = true)
 |-- sintoma_dificuldade_respirar: string (nullable = true)
 |-- sintoma_dor_cabeca: string (nullable = true)
 |-- sintoma_dor_peito: string (nullable = true)
 |-- sintoma_nausea: string (nullable = true)
 |-- sintoma_nariz_entupido: string (nullable = true)
 |-- sintoma_fadiga: string (nullable = true)
 |-- sintoma_dor_olhos: string (nullable = true)
 |-- sintoma_perda_cheiro_sabor: string (nullable = true)
 |-- sintoma_dor_muscular: string (nullable = true)
 |-- sintoma_diarreia: string (nullable = true)
 |-- providencia_ficou_em_casa: string (nullable = true)
 |-- providencia_ligou_medico: string (nullable = true)
 |-- providencia_comprou_remedio: string (nullable = true)
 |-- providencia_visita_sus: string (nullable = true)
 |-- providencia_visita_privado: string 

# Exportando camada silver

In [11]:
df_silver.write.mode("overwrite").parquet("s3://rhgc-tech-challenge-03/output/2-silver/")

# Camada gold
# Agregações

## Cuidado domiciliar

In [12]:
df_gold = df_gold.withColumn(
    "cuidado_domiciliar",
    F.when((F.col("providencia_ficou_em_casa") == 1) |
           (F.col("providencia_ligou_medico") == 1) |
           (F.col("providencia_comprou_remedio") == 1) |
           (F.col("providencia_visita_sus") == 1) |
           (F.col("providencia_visita_privado") == 1), "Sim").otherwise("Não")
)

NameError: name 'df_gold' is not defined


## Buscou atendimento

In [ ]:
df_gold = df_gold.withColumn(
    "buscou_atendimento",
    F.when(
        (F.col("buscou_posto_sus") == 1) |
        (F.col("buscou_pronto_socorro_sus") == 1) |
        (F.col("buscou_hospital_sus") == 1) |
        (F.col("buscou_ambulatorio_privado") == 1) |
        (F.col("buscou_pronto_socorro_privado") == 1) |
        (F.col("buscou_hospital_privado") == 1),
        "Sim"
    ).otherwise("Não")
)

## Tipo de estabelecimento

In [ ]:
# Condições lógicas para cada tipo de rede
foi_sus = (F.col("buscou_posto_sus") == 1) | (F.col("buscou_pronto_socorro_sus") == 1) | (F.col("buscou_hospital_sus") == 1)
foi_privado = (F.col("buscou_ambulatorio_privado") == 1) | (F.col("buscou_pronto_socorro_privado") == 1) | (F.col("buscou_hospital_privado") == 1)

# Aplicação da agregação no DataFrame
df_gold = df_gold.withColumn(
    "tipo_estabelecimento",
    F.when(foi_sus & foi_privado, "Ambos")
     .when(foi_sus, "SUS")
     .when(foi_privado, "Privado")
     .otherwise("Não aplicável")
)

## Testagem

In [ ]:
# 1. Mapeamento das condições lógicas baseadas no dicionário do IBGE
testou_positivo = (F.col("resultado_swab") == 1) | (F.col("resultado_teste_rapido") == 1) | (F.col("resultado_exame_sangue") == 1)
aguardando_resultado = (F.col("resultado_swab") == 4) | (F.col("resultado_teste_rapido") == 4) | (F.col("resultado_exame_sangue") == 4)
testou_negativo = (F.col("resultado_swab") == 2) | (F.col("resultado_teste_rapido") == 2) | (F.col("resultado_exame_sangue") == 2)

# 2. Criação da nova coluna agregada
df_gold = df_gold.withColumn(
    "status_testagem",
    F.when(F.col("realizou_teste") == 2, "Não testado")
     .when(testou_positivo, "Positivo")
     .when(aguardando_resultado, "Aguardando resultado")
     .when(testou_negativo, "Negativo")
     .when(F.col("realizou_teste") == 1, "Inconclusivo / Sem resultado")
     .otherwise("Não aplicável")
)

## Auxílios recebidos

In [ ]:
df_gold = df_gold.withColumn(
    "auxilio_social",
    F.when((F.col("auxilio_emergencial") == 1) | (F.col("seguro_desemprego") == 1), "Sim").otherwise("Não")
)

## Afastamento do trabalho

In [ ]:
df_gold = df_gold.withColumn(
    "afastamento_trabalho",
    F.when((F.col("motivo_afastamento_temporario") == 2), "Não").otherwise(F.col("continuou_remunerado").cast(StringType()))
)

## Grupos de risco

In [ ]:
df_gold = df_gold.withColumn(
    "grupo_risco",
    F.when(
        (F.column("comorbidade_diabetes") == 1) |
        (F.column("comorbidade_hipertensao") == 1) |
        (F.column("comorbidade_asma") == 1) |
        (F.column("comorbidade_doenca_cardiaca") == 1) |
        (F.column("comorbidade_depressao") == 1) |
        (F.column("comorbidade_cancer") == 1),
        "Sim"
    ).otherwise("Não")
)

## Remoção das colunas utilizadas nas agregações

In [ ]:
colunas_para_remover = [
    # Medidas e Atendimento (originais que foram agregadas ou não são mais necessárias)
    "providencia_ficou_em_casa", "providencia_ligou_medico", "providencia_comprou_remedio",
    "providencia_visita_sus", "providencia_visita_privado", "providencia_outra",
    "buscou_posto_sus", "buscou_pronto_socorro_sus", "buscou_hospital_sus",
    "buscou_ambulatorio_privado", "buscou_pronto_socorro_privado", "buscou_hospital_privado",

    # Comorbidades (originais que foram agregadas em 'grupo_risco')
    "comorbidade_diabetes", "comorbidade_hipertensao", "comorbidade_asma",
    "comorbidade_doenca_cardiaca", "comorbidade_depressao", "comorbidade_cancer",

    # Testagem (originais que foram agregadas em 'status_testagem')
    "realizou_teste", "resultado_swab", "resultado_teste_rapido", "resultado_exame_sangue",

    # Afastamento e Auxílios (originais que foram agregadas em 'afastamento_trabalho' e 'auxilio_social')
    "motivo_afastamento_temporario", "continuou_remunerado",
    "auxilio_emergencial", "seguro_desemprego"
]

for coluna in colunas_para_remover:
    if coluna in df_gold.columns:  # Verifica se a coluna existe antes de tentar remover
        df_gold = df_gold.drop(coluna)

# Mapping dos valores das colunas

In [ ]:
# Mappings para colunas que ainda são do tipo inteiro (ou cujos códigos são inteiros)
sexo_mapping = {
    1: "Masculino",
    2: "Feminino"
}

cor_raca_mapping = {
    1: "Branca",
    2: "Preta",
    3: "Amarela",
    4: "Parda",
    5: "Indígena",
    9: "Ignorado"
}

escolaridade_mapping = {
    1: "Sem instrução",
    2: "Fundamental incompleto ou equivalente",
    3: "Fundamental completo ou equivalente",
    4: "Médio incompleto ou equivalente",
    5: "Médio completo ou equivalente",
    6: "Superior incompleto ou equivalente",
    7: "Superior completo ou equivalente",
    8: "Pós-graduação"
}

plano_saude_mapping = {
    1: "Sim",
    2: "Não",
    9: "Ignorado"
}

nivel_isolamento_mapping = {
    1: "Não fez restrição, levou vida normal como antes da pandemia",
    2: "Reduziu o contato com as pessoas, mas continuou saindo de casa para trabalho ou atividades não essenciais e/ou recebendo visitas",
    3: "Ficou em casa e só saiu em caso de necessidade básica",
    4: "Ficou rigorosamente em casa",
    9: "Ignorado"
}

sintoma_mapping = {
    1: "Sim",
    2: "Não",
    3: "Não sabe",
    9: "Ignorado"
}

# Mappings para colunas que se tornaram do tipo string devido ao preenchimento anterior de nulos
# Seus valores e chaves de mapeamento devem ser strings.
internacao_mapping = {
    "1": "Sim",
    "2": "Não",
    "3": "Não foi atendido",
    "9": "Ignorado"
}

suporte_ventilatorio_mapping = {
    "1": "Sim",
    "2": "Não",
    "9": "Ignorado"
}

status_trabalho_mapping = {
    "1": "Sim",
    "2": "Não"
}

trabalho_remoto_mapping = {
    "1": "Sim",
    "2": "Não"
}

# Função para aplicar mapeamento textual a uma coluna usando uma cadeia 'when'
def apply_textual_mapping(df, column_name, mapping, default_value="Não aplicável"):
    # Inicia a cadeia 'when' com a primeira entrada do mapeamento.
    # Assume que o mapeamento não estará vazio para os casos fornecidos.
    keys = list(mapping.keys())
    first_key = keys[0]
    first_value = mapping[first_key]

    expression = F.when(F.col(column_name) == first_key, first_value)

    # Adiciona os mapeamentos subsequentes à cadeia
    for i in range(1, len(keys)):
        key = keys[i]
        value = mapping[key]
        expression = expression.when(F.col(column_name) == key, value)

    # Adiciona uma cláusula 'otherwise' final para quaisquer valores não mapeados explicitamente,
    # incluindo possíveis nulos originais não capturados por 'Não aplicável'.
    expression = expression.otherwise(default_value)

    return df.withColumn(column_name, expression)

# Aplica os mapeamentos ao df_gold
df_gold = apply_textual_mapping(df_gold, "internacao", internacao_mapping)
df_gold = apply_textual_mapping(df_gold, "suporte_ventilatorio", suporte_ventilatorio_mapping)
df_gold = apply_textual_mapping(df_gold, "plano_saude", plano_saude_mapping)
df_gold = apply_textual_mapping(df_gold, "nivel_isolamento", nivel_isolamento_mapping)
df_gold = apply_textual_mapping(df_gold, "sexo", sexo_mapping)
df_gold = apply_textual_mapping(df_gold, "cor_raca", cor_raca_mapping)
df_gold = apply_textual_mapping(df_gold, "escolaridade", escolaridade_mapping)
df_gold = apply_textual_mapping(df_gold, "status_trabalho", status_trabalho_mapping)
df_gold = apply_textual_mapping(df_gold, "trabalho_remoto", trabalho_remoto_mapping)

# Lista de colunas de sintomas
colunas_sintomas = ["sintoma_febre", "sintoma_tosse", "sintoma_dor_garganta", "sintoma_dificuldade_respirar", "sintoma_dor_cabeca",
                    "sintoma_dor_peito", "sintoma_nausea", "sintoma_nariz_entupido", "sintoma_fadiga", "sintoma_dor_olhos",
                    "sintoma_perda_cheiro_sabor", "sintoma_dor_muscular", "sintoma_diarreia"]

# Aplica o mapeamento para cada coluna de sintoma
for sintoma_col in colunas_sintomas:
    df_gold = apply_textual_mapping(df_gold, sintoma_col, sintoma_mapping)


In [ ]:
df_gold.printSchema()

# Tratamento de nulidades

In [ ]:
df_gold.show(1)

In [ ]:
for column in df_gold.columns:
    null_count = df_gold.filter(F.col(column).isNull()).count()
    if null_count > 0:
        print(f"Coluna '{column}': {null_count} valores nulos")

In [ ]:
columns_with_nulls = [
    "internacao",
    "suporte_ventilatorio",
    "status_trabalho",
    "trabalho_remoto",
    "afastamento_trabalho"
]

for column_name in columns_with_nulls:
    if column_name in df_gold.columns:
        # Preenche valores nulos com 'Não aplicável' para as colunas especificadas.
        # Assume que as colunas já foram tratadas para serem do tipo String, ou que `fillna` pode lidar com elas diretamente.
        df_gold = df_gold.na.fill("Não aplicável", subset=[column_name])

print("Valores nulos preenchidos em colunas selecionadas com 'Não aplicável'.")

# Ordenação das colunas

In [ ]:
colunas_ordenadas = [
    # Perfil Demográfico
    "idade", "sexo", "cor_raca",

    # Socioeconómico e Trabalho
    "escolaridade", "faixa_renda", "status_trabalho",
    "afastamento_trabalho", "trabalho_remoto", "auxilio_social",

    # Histórico e Prevenção
    "grupo_risco", "plano_saude", "nivel_isolamento",

    # Quadro Clínico (Sintomas)
    "sintoma_febre", "sintoma_tosse", "sintoma_dor_garganta", "sintoma_dificuldade_respirar", "sintoma_dor_cabeca",
    "sintoma_dor_peito", "sintoma_nausea", "sintoma_nariz_entupido", "sintoma_fadiga", "sintoma_dor_olhos",
    "sintoma_perda_cheiro_sabor", "sintoma_dor_muscular", "sintoma_diarreia",

    # Ações e Desfechos Clínicos
    "cuidado_domiciliar", "buscou_atendimento", "tipo_estabelecimento",
    "status_testagem", "internacao", "suporte_ventilatorio"
]

# Aplicar a seleção e reordenação
df_gold = df_gold.select(colunas_ordenadas)

# Exportando o arquivo final da camada gold

In [ ]:
df_gold.write.mode("overwrite").parquet("s3://rhgc-tech-challenge-03/output/3-gold/")